# Homework 02: Formula 1 Data Analysis with PySpark

Author: Rongshan Wei 

UNI: rw3082

Date: April 6th, 2026

## Project Overview
This notebook contains the analysis of the Formula 1 (F1) historical dataset using PySpark on the Databricks platform. The primary goal of this exercise is to leverage distributed computing to process large-scale sports data, performing complex aggregations, feature engineering, and historical performance tracking.

## Technical Stack
* Platform: Databricks (Cloud-based Spark environment)
* Engine: Apache Spark (PySpark SQL and Window Functions)
* Version Control: GitHub (Frequent commits to track development progress)

## Dataset Description
The analysis utilizes the comprehensive F1 dataset, which includes several relational tables:
* drivers: Personal information and unique identifiers for all F1 drivers.
* results: Finishing positions, points, and status for every race.
* pit_stops: Granular data on every pit stop event, including duration.
* races: Contextual information about each Grand Prix (date, location, year).

## Methodology & Best Practices
In accordance with the assignment requirements, each task in this notebook is structured as follows:
* Logic Formulation: A clear Markdown explanation of the analytical approach.
* Implementation: Optimized PySpark code to process the data.
* Code Walkthrough: A detailed explanation of specific Spark commands and their roles in achieving the result.
* Result Interpretation: A detailed explanation of results insights.

## I. Global Configuration & Environment
Before processing large-scale datasets, it is a Best Practice to document the computational environment to ensure reproducibility.
* Spark Version: 3.x (Databricks Runtime)
* Cluster Configuration: Standard Runtime
* Primary Language: PySpark (Python 3.10+)
* Version Control: Integrated Git Folders

## II. Library Ingestion

1. Logic Formulation

Following PEP 8 guidelines, all imports are centralized at the top of the notebook to manage dependencies effectively. I utilize the `pyspark.sql.functions` module aliased as `F` to maintain a clean namespace and prevent shadowing Python's built-in functions.

2. Implementation

In [0]:
# Standard Library Imports
import sys

# Third-party Library Imports: PySpark SQL
from pyspark.sql import functions as F
from pyspark.sql.window import Window

3. Code Walkthrough
* `Window`: This module is essential for Tasks 2 and 5, allowing us to perform calculations across a set of rows related to the current row (e.g., ranking and cumulative sums).
* Namespace Aliasing: Using `F` is a common industry standard in Spark development that balances brevity with clarity.

## III. Data Ingestion & Schema Discovery

1. Logic Formulation

To satisfy the Computational Thinking principle of "Decomposition," I separate data loading from analysis. I use the `inferSchema=True` parameter to ensure Spark correctly identifies integers and timestamps. After loading, I will utilize Databricks' native visual engine to audit the data types, ensuring the "inputs" match the "expectations" of our analytical functions.

2. Implementation

In [0]:
# Configuration: Define root path for abstraction
VOLUME_PATH = '/Volumes/gr5069/raw/f1_data/'

# Load relational F1 tables with Schema Inference
# ----------------------------------------------------------------------
df_pit_stops = spark.read.csv(f'{VOLUME_PATH}pit_stops.csv', header=True, inferSchema=True)
df_drivers   = spark.read.csv(f'{VOLUME_PATH}drivers.csv', header=True, inferSchema=True)
df_results   = spark.read.csv(f'{VOLUME_PATH}results.csv', header=True, inferSchema=True)
df_races     = spark.read.csv(f'{VOLUME_PATH}races.csv', header=True, inferSchema=True)

# Interactive Schema & Data Audit
print("### Schema & Data Audit: Drivers Dataset ###")
display(df_drivers)

print("### Schema & Data Audit: Pit Stops Dataset ###")
display(df_pit_stops)

3. Code Walkthrough
* `display()` for Profiling: In modern Databricks notebooks, running `display(df)` automatically generates a UI where you can click the "Data Profile" tab (or the "+" sign next to the table view). This provides the same histograms, null counts, and descriptive statistics that `summarize()` would have shown, but it is fully supported on Serverless clusters.

* `inferSchema=True`: This remains crucial. For the upcoming Task 4, we need `dob` to be a `date` type and milliseconds to be an `integer` to avoid casting errors.

---

## IV. Analytical Tasks

### Task 1: Pit Stop Statistical Metrics

1. Logic Formulation

The objective is to calculate the average, fastest (minimum), and slowest (maximum) pit stop times for each driver in every individual race.

Based on the initial schema discovery, the `duration` column is stored as a `string`, which is unsafe for mathematical aggregations. To ensure computational accuracy and robustness, I will use the numerical `milliseconds` column, convert it into seconds (by dividing by 1000), and apply the aggregation functions (`avg`, `min`, `max`). The dataset will be grouped by a composite key of `raceId` and `driverId` to maintain the correct level of granularity.

2. Implementation

In [0]:
# Task 1: Calculate pit stop statistics with precision formatting
pit_stop_stats = (
    df_pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        # Convert milliseconds to seconds and round to 3 decimal places
        F.round(F.avg("milliseconds") / 1000, 3).alias("avg_pit_time_sec"),
        F.round(F.min("milliseconds") / 1000, 3).alias("fastest_pit_time_sec"),
        F.round(F.max("milliseconds") / 1000, 3).alias("slowest_pit_time_sec")
    )
    .orderBy("raceId", "driverId")
)

# Render the dataframe using Databricks interactive UI
display(pit_stop_stats)

3. Code Walkthrough
* `F.round(..., 3)`: This function resolves the floating-point precision issue (e.g., transforming 24.145666... into a clean 24.146). It is a standard data engineering practice to format numerical outputs before presenting them to stakeholders.

4. Result Interpretation

The output normalizes the diverse pit stop durations into a standardized seconds format. A quick visual audit reveals that average pit stops generally cluster around the 22-25 second mark. By extracting the fastest and slowest times, we can immediately identify outlier events (e.g., severe pit lane delays) versus the baseline efficiency of a team's pit crew during a specific Grand Prix.

---

### Task 2: Rank Average Pit Stop by Finishing Position

1. Logic Formulation

To analyze whether faster pit stops correlate with better race finishes, I will join the results dataset with the pit_stop_stats calculated in Task 1.

Handling Missing Data & DNFs: 
* I will perform an inner join. This naturally filters out historical races (pre-2011) where pit stop data was never recorded, resolving the issue of null averages.
* For drivers who Did Not Finish (DNF), I have decided to include them in the pit stop ranking. Even if a driver crashed later in the race, their completed pit stops remain valid indicators of the pit crew's efficiency. They will appear lower in the final sorting due to their high positionOrder.

2. Implementation

In [0]:
# Task 2: Rank pit stops and order by finishing position
# Define a window to rank average pit times within each specific race (1 = fastest)
pit_window = Window.partitionBy("raceId").orderBy("avg_pit_time_sec")

task_2_ranked = (
    df_results.select("raceId", "driverId", "positionOrder", "positionText")
    # Use INNER join to automatically drop pre-2011 races
    .join(
        pit_stop_stats.select("raceId", "driverId", "avg_pit_time_sec"),
        on=["raceId", "driverId"],
        how="inner"
    )
    # Apply the ranking function over our defined window
    .withColumn("pit_efficiency_rank", F.rank().over(pit_window))
    # Final sort: by race, then by how they finished the race
    .orderBy("raceId", "positionOrder")
)

# Interactive profiling to verify the logic
display(task_2_ranked)

3. Code Walkthrough
* `how="inner"`: Acts as an automatic filter. If `raceId=18` has no pit data, it simply won't appear in this analysis, ensuring we only evaluate complete data pairs.
* `Window.partitionBy("raceId")`: This resets the ranking for every new race. We don't want to compare a pit stop in Monaco with one in Monza.
* `F.rank().over(pit_window)`: Assigns a ranking (1, 2, 3...) based on the `avg_pit_time_sec`.

4. Result Interpretation

By aligning pit efficiency rankings with the final race `positionOrder`, the data highlights a nuanced reality of F1: while a lightning-fast pit stop does not guarantee a podium finish (as seen by lower-ranked drivers occasionally having the fastest pit times), catastrophic pit stops almost always negatively impact the final standing. The inclusion of DNF drivers ensures our pit crew efficiency metrics remain unbiased by on-track accidents.

---

### Task 3: Impute Missing Driver Codes

1. Logic Formulation
The goal of this task is to standardize the `drivers` dataset by inserting a 3-letter abbreviation for drivers who are missing an official broadcast code.

Domain Knowledge & Strategy:
* Official 3-letter timing codes are a relatively modern addition to F1. Historical drivers typically lack this data. In raw CSV datasets, missing values are often represented either as native nulls or as the string literal `"\N"`.
* To impute these missing values, I will use conditional logic. If a driver's code is missing or invalid, I will dynamically generate one by extracting the first three letters of their `surname` and converting them to uppercase (e.g., "Senna" becomes "SEN"). If a valid code already exists, I will preserve the official record.

2. Implementation

In [0]:
# Task 3: Impute missing codes using string manipulation and conditional logic
task_3_drivers = (
    df_drivers
    # We overwrite the existing 'code' column with our corrected logic
    .withColumn(
        "code",
        F.when(
            # Condition: Check for actual nulls OR the string literal "\N"
            F.col("code").isNull() | (F.col("code") == "\\N"), 
            # Action if true: take first 3 chars of surname and capitalize
            F.upper(F.substring(F.col("surname"), 1, 3))
        )
        # Action if false: keep the original valid code
        .otherwise(F.col("code"))
    )
)

# Verification: Display historical drivers (low driverId) to prove the imputation worked
print("--- Task 3: Driver Codes Imputation Verification ---")
display(
    task_3_drivers
    .select("driverId", "forename", "surname", "code", "nationality")
    .orderBy("driverId") # Historical drivers appear first and usually have missing codes
)

3. Code Walkthrough
* `F.when(...).otherwise(...)`: This is PySpark's equivalent of an `IF-ELSE` statement or SQL's `CASE WHEN`. It is a highly optimized way to perform row-level conditional transformations without using slow Python loops.
* `F.col("code") == "\\N"`: A critical data-cleaning step. Many raw CSV exports use `\N` to denote nulls. Checking for this explicitly prevents our logic from failing on dirty data.
* `F.upper(F.substring(F.col("surname"), 1, 3))`: I nest two string functions here. `substring` extracts exactly 3 characters starting from index 1 (PySpark is 1-indexed for SQL functions), and `upper` ensures the code matches the visual formatting of official F1 timing screens.

4. Result Interpretation

The imputation logic successfully backfilled legacy records, standardizing the dataset. Historical icons from the 1950s and 60s who previously possessed null or `\N` values now feature clean, 3-letter uppercase identifiers derived from their surnames (e.g., "FAR" for Farina). This ensures downstream visualization tools or dashboards can reliably use the `code` column without breaking.

---

### Task 4: Calculate Driver Age & Age Extremes

1. Logic Formulation
To accurately calculate a driver's age at the time of a specific race, I will join the `drivers` dataset with the `races` dataset.

The Age Calculation Logic: Using PySpark's `datediff` function, I will find the total number of days between the driver's date of birth (`dob`) and the race date. Dividing this by 365 and applying the `floor` function will yield their age in completed years.

Finding Extremes:
I will then use Window Functions partitioned by `raceId` to identify the minimum and maximum ages within each Grand Prix. This allows us to flag the oldest and youngest participants for every historical event.

2. Implementation

In [0]:
# Task 4: Calculate age at race date and find oldest/youngest drivers
# 1. Join drivers with races and results to get the date context
driver_age_df = (
    df_results.select("raceId", "driverId")
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), on="driverId")
    .join(df_races.select("raceId", "date"), on="raceId")
    .withColumn(
        "age_at_race", 
        F.floor(F.datediff(F.col("date"), F.col("dob")) / 365)
    )
)

# 2. Define a Window to find extremes per race
race_window = Window.partitionBy("raceId")

# 3. Add columns for the oldest and youngest age in that race
task_4_final = (
    driver_age_df
    .withColumn("youngest_age_in_race", F.min("age_at_race").over(race_window))
    .withColumn("oldest_age_in_race", F.max("age_at_race").over(race_window))
    .select("raceId", "forename", "surname", "age_at_race", "youngest_age_in_race", "oldest_age_in_race")
    .orderBy("raceId", "age_at_race")
)

display(task_4_final)

3. Code Walkthrough
* `F.datediff(F.col("date"), F.col("dob"))`: This subtracts the two date columns to get the absolute age in days.
`* F.min().over(race_window)`: Instead of a traditional `groupBy`, using a `Window` allows us to keep all individual driver rows while appending the race-wide extreme values as new columns for easy comparison.

4. Result Interpretation

The age derivation reveals the demographic shifts in Formula 1 over the decades. The data contrasts the modern era—where phenoms often debut in their late teens or early twenties—against the sport's early years, where the "oldest" drivers in a race were frequently well into their 40s or 50s. The contextual `datediff` accurately captures the exact age on race day, preventing chronological drift.

---

### Task 5: Cumulative Podium Finishes

1. Logic Formulation

To determine the cumulative podiums for each driver chronologically, I will use a Window Function spanning from `unboundedPreceding` to `currentRow`.

To ensure the final dataset is human-readable and presentation-ready, I will expand the join logic to include the drivers table. This allows us to map the opaque `driverId` to the actual `forename` and `surname`, making the historical tracking immediately understandable during the data audit.

2. Implementation

In [0]:
# Task 5: Calculate cumulative wins, 2nd places, and 3rd places per driver
# 1. Define the chronological window for each driver's career progression
career_window = (
    Window.partitionBy("driverId")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# 2. Join tables and apply cumulative aggregations
task_5_podiums = (
    df_results.select("raceId", "driverId", "positionOrder")
    # Join races to get the chronological 'date'
    .join(df_races.select("raceId", "date"), on="raceId")
    # ---> NEW: Join drivers to get human-readable names <---
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId")
    .withColumn(
        "cumulative_wins", 
        F.sum(F.when(F.col("positionOrder") == 1, 1).otherwise(0)).over(career_window)
    )
    .withColumn(
        "cumulative_2nd_places", 
        F.sum(F.when(F.col("positionOrder") == 2, 1).otherwise(0)).over(career_window)
    )
    .withColumn(
        "cumulative_3rd_places", 
        F.sum(F.when(F.col("positionOrder") == 3, 1).otherwise(0)).over(career_window)
    )
    # Sort by driver and date to ensure logical chronological progression
    .orderBy("driverId", "date")
)

# 3. Present the data with names clearly visible
print("--- Task 5: Cumulative Podium Tracking ---")
display(
    task_5_podiums
    # Bringing forename and surname to the front for immediate context
    .select(
        "driverId", "forename", "surname", "raceId", "date", 
        "positionOrder", "cumulative_wins", "cumulative_2nd_places", "cumulative_3rd_places"
    )
)

3. Code Walkthrough

* `.join(df_drivers...)`: By joining the drivers table and explicitly selecting `forename` and `surname`, I decouple the analysis from reliance on metadata lookups.

* `.select(...)` Reordering: In the `display()` function, I moved the names right next to the `driverId`. This UI/UX tweak makes it much easier to scroll through and visually track a specific driver's career arc without jumping your eyes across the screen.

4. Result Interpretation

The unbounded chronological window provides a longitudinal view of driver dominance. By tracking the cumulative podiums, we can visualize the exact trajectory of a driver's career—pinpointing breakout seasons where their `cumulative_wins` rapidly spiked, versus dormant periods. Appending the driver names makes this historical tracking immediately comprehensible for business stakeholders.

---

### Task 6: Calculate Total Career Points

1. Logic Formulation
The final objective is to calculate the all-time total points scored by each driver across their entire F1 career.

Because this is an absolute total rather than a running metric or an event-specific extreme, Window Functions is not needed. I can return to the foundational Map-Reduce / GroupBy paradigm. I will group the `results` table by `driverId` and calculate the sum of the `points` column. Following the established Best Practice of producing human-readable deliverables, I will join these aggregated results back to the `drivers` table to include the `forename` and `surname`, and finally sort the dataframe in descending order to create an all-time leaderboard.

2. Implementation

In [0]:
# Task 6: Calculate all-time career points for each driver
task_6_points = (
    df_results
    # Grouping by driver to collapse all historical race records into a single row per driver
    .groupBy("driverId")
    .agg(
        # Summing the 'points' column to get the career total
        F.sum("points").alias("total_career_points")
    )
    # Joining with the drivers table to make the output presentation-ready
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId")
    # Reordering columns for UX and sorting from highest to lowest points
    .select("driverId", "forename", "surname", "total_career_points")
    .orderBy(F.col("total_career_points").desc())
)

# Display the final leaderboard
print("--- Task 6: All-Time Driver Points Leaderboard ---")
display(task_6_points)

3. Code Walkthrough
* `groupBy("driverId")`: This condenses the massive results table (tens of thousands of rows) down to roughly ~850 unique driver rows.
* `F.sum("points")`: F1 has historically awarded half-points in certain shortened races (e.g., 1984 Monaco, 2021 Spa). Because `points` in the raw dataset is usually cast as a float, `F.sum()` will safely handle and aggregate these decimal values without losing precision.
* `.desc()`: Sorting in descending order instantly turns this output into an All-Time Hall of Fame leaderboard, which is exactly what an analyst or stakeholder would want to see when asking for "total points."

4. Result Interpretation

The final aggregation presents the all-time F1 Hall of Fame based on total career points. However, an analytical caveat is necessary when reading this leaderboard: modern drivers (like Lewis Hamilton) are inherently favored. This is because the FIA points system has inflated over time (e.g., a win today awards 25 points, whereas historically it awarded 10 or 9). Nonetheless, it accurately reflects total aggregate scoring under the rules of their respective eras.

---

## V. Project Conclusion & Teardown

### Executive Summary
This notebook successfully constructed a robust ETL and analytical pipeline for the historical Formula 1 dataset. Adhering to strict computational thinking principles and coding etiquette, the analysis transitioned from raw CSV ingestion to advanced PySpark transformations. 

### Key Technical Achievements:

1. **Schema Integrity:** Enforced strict data types, ensuring dates and milliseconds were mathematically viable for aggregations.
2. **Advanced Windowing:** Leveraged PySpark `Window` functions for both partitioned calculations (e.g., race extremes) and chronological running totals (e.g., cumulative podiums).
3. **Data Cleansing:** Deployed fault-tolerant conditional logic (`F.when`) to seamlessly impute missing historical broadcast codes without compromising official modern records.
4. **Optimization:** Minimized expensive shuffles by prioritizing optimal Join strategies and utilizing Databricks' native `display()` for memory-efficient, interactive data profiling.

*End of Assignment. All Spark jobs executed successfully.*